# Reinforcement Learning Assignment: RoboRacer in Gazebo

## Assignment Introduction

In this assignment, we use reinforcement learning to make a simulated RoboRacer car drive around a track in Gazebo.

The available input topics are:

- `/scan`: LiDAR scan data
- `/odom`: raw odometry data

The output topic is:

- `/cmd_vel`: velocity command sent to the robot

The goal is to train an RL agent that can drive the robot and complete a lap on the track.

In this assignment:

- the **agent** is the RoboRacer car,
- the **environment** is the Gazebo simulation,
- the **observation** is built from `/scan` and `/odom`,
- the **action** is converted into a command on `/cmd_vel`,
- the **reward function** tells the agent whether its driving behavior is good or bad,
- the **value function / action-value function** helps the agent estimate which action is better,
- the **training script** connects the environment with the learning algorithm.

The main TODOs are:

1. Write and test a reward function.
2. Understand the action-value function Q(s, a) used by DQN.
3. Create and run a training script.

## Short Theory Note from Reinforcement Learning

This section gives the minimum theory needed before working with the RoboRacer RL package.

The theory is based mainly on:

- Sutton and Barto, *Reinforcement Learning: An Introduction*
- OpenAI Spinning Up, *Key Concepts in RL*
- Gymnasium environment API documentation

### Agent and Environment

In reinforcement learning, an agent interacts with an environment. At each time step, the agent receives information from the environment, selects an action, and receives a reward.

For this assignment:

| RL term | RoboRacer meaning |
|---|---|
| Agent | RoboRacer car |
| Environment | Gazebo simulation |
| Observation | Vector built from `/scan` and `/odom` |
| Action | Driving decision selected by the policy |
| Output command | `/cmd_vel` |
| Reward | Score for the last driving action |
| Policy | Function/neural network that maps observations to actions |

### Observation

An observation is the information available to the agent at the current time step.

In our case, the robot does not receive the complete Gazebo world. It only receives sensor-based information. The observation vector is constructed from LiDAR and odometry data.

### Action

An action is the decision selected by the agent.

For a simple DQN controller, the action space can be discrete, for example:

```text
Action 0 → turn left
Action 1 → go straight
Action 2 → turn right
```

The selected action is then converted into a `/cmd_vel` message.

For `/cmd_vel`, the important fields are usually:

```text
linear.x   → forward speed
angular.z  → turning / yaw command
```

### Reward

A reward is a scalar feedback signal. It tells the agent how good or bad the last action was.

For RoboRacer:

```text
safe forward driving → positive reward
unsafe driving       → negative reward
crash                → large negative reward
```

The reward function is very important because the agent does not directly know what “good driving” means. It learns whatever behavior the reward function encourages.

### Return

The return is the total future reward that the agent tries to maximize.

A common discounted return is:

```text
G_t = R_{t+1} + gamma R_{t+2} + gamma^2 R_{t+3} + ...
```

Here, `gamma` is the discount factor. If `gamma` is close to 1, future rewards are considered important. If `gamma` is small, the agent focuses more on immediate rewards.

### Value Function

A value function estimates how good a situation is.

```text
V(s) = expected future return from state s
```

In simple words:

```text
How good is this current situation?
```

### Action-Value Function

An action-value function estimates how good it is to take a specific action in a specific state.

```text
Q(s, a) = expected future return after taking action a in state s
```

In simple words:

```text
If I am in this situation and take this action, how good will it be?
```

DQN learns an approximation of this action-value function using a neural network.

### Training

Training means repeatedly allowing the agent to interact with the environment and improve its policy.

The basic loop is:

```text
1. Read observation
2. Select action
3. Publish command to `/cmd_vel`
4. Receive next observation
5. Compute reward
6. Update the learning algorithm
7. Repeat
```

## System Check: ROS 2 Package Visibility and Active Topics

This section is only a quick check. It is not one of the main coding tasks.

Before running the notebook, the Gazebo simulation should be launched in a separate terminal. Also make sure the Gazebo simulation is started/unpaused, because it is usually paused after launch.

The expected important topics are:

```text
/scan
/odom
/cmd_vel
```

If these topics are available, the RL package can receive sensor data and publish driving commands.

In [ ]:
import subprocess

def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

print("RoboRacer packages:")
run_cmd("ros2 pkg list | grep roboracer")

print("\nActive topics:")
run_cmd("ros2 topic list")

print("\nTopic information:")
run_cmd("ros2 topic info /scan")
run_cmd("ros2 topic info /odom")
run_cmd("ros2 topic info /cmd_vel")

## Observation Vector Explanation

The RL agent cannot directly understand ROS messages. Therefore, the information from `/scan` and `/odom` must be converted into a numerical vector.

This vector is called the observation vector.

### From `/scan` to LiDAR sectors

The `/scan` topic contains LiDAR distance measurements around the robot. A raw LiDAR scan may contain many values, so we reduce it into sectors.

Each sector summarizes one angular region around the car.

For a simple 9-sector explanation:

| Sector | Region around car | Meaning |
|---|---|---|
| Sector 1 | Far left | Detects obstacles or free space on the far-left side |
| Sector 2 | Left | Detects free space on the left side |
| Sector 3 | Front-left | Helps with left turning and front-left obstacle avoidance |
| Sector 4 | Slightly left of front | Detects obstacles near the front-left direction |
| Sector 5 | Directly front | Most important for immediate collision risk |
| Sector 6 | Slightly right of front | Detects obstacles near the front-right direction |
| Sector 7 | Front-right | Helps with right turning and front-right obstacle avoidance |
| Sector 8 | Right | Detects free space on the right side |
| Sector 9 | Far right | Detects obstacles or free space on the far-right side |

### From `/odom` to motion information

The `/odom` topic gives information about the motion of the robot. From odometry, the environment can use values such as speed.

This helps the agent understand not only where obstacles are, but also how the car is moving.

### Final observation idea

The observation vector can be understood as:

```text
observation = LiDAR sector values + odometry/motion values + optional previous action information
```

This observation vector becomes the input to the RL policy or neural network.

## TODO 1: Reward Function

The reward function defines what the robot should learn.

A good reward function should encourage:

- moving forward,
- staying away from walls,
- smooth steering,
- avoiding crashes.

It should penalize:

- collisions,
- driving too close to obstacles,
- unnecessary oscillation,
- stopping without reason.

In the RoboRacer package, the reward function is located in:

```text
roboracer_rl/gym_env.py
```

Look for:

```text
_reward()
```

Before editing the real package, test your reward idea below.

In [ ]:
def student_reward(speed, min_lidar_distance, steering, crashed):
    reward = 0.0

    # Encourage forward movement
    reward += speed

    # Penalize driving close to obstacles
    if min_lidar_distance < 0.5:
        reward -= 2.0

    # Penalize aggressive steering
    reward -= 0.1 * abs(steering)

    # Penalize crash strongly
    if crashed:
        reward -= 50.0

    return reward


test_cases = [
    {"speed": 1.0, "min_lidar_distance": 2.0, "steering": 0.0, "crashed": False},
    {"speed": 1.0, "min_lidar_distance": 0.3, "steering": 0.2, "crashed": False},
    {"speed": 0.2, "min_lidar_distance": 0.1, "steering": 0.8, "crashed": True},
]

for case in test_cases:
    print(case, "reward =", student_reward(**case))

After testing, modify the actual `_reward()` function in `gym_env.py`.

Then train the agent and observe how the car behavior changes.

## TODO 2: Action-Value Function Q(s, a)

DQN is based on the action-value function.

The action-value function is written as:

```text
Q(s, a)
```

It means:

```text
How good is action a in state s?
```

For example, assume the robot has three possible actions:

```text
0 → turn left
1 → go straight
2 → turn right
```

The DQN network estimates one Q-value for each action. The action with the highest Q-value is selected.

In [ ]:
# Example Q-values predicted by a DQN-like model for one observation

actions = {
    0: "turn left",
    1: "go straight",
    2: "turn right"
}

q_values = {
    0: 2.1,
    1: 4.8,
    2: 1.5
}

best_action = max(q_values, key=q_values.get)

print("Estimated Q-values:")
for action_id, value in q_values.items():
    print(f"Action {action_id} ({actions[action_id]}): Q = {value}")

print("\nSelected action:")
print(f"Action {best_action}: {actions[best_action]}")

This is the basic idea behind DQN:

```text
observation → neural network → Q-values for all actions → choose best action
```

In the real training script, the neural network learns these Q-values from repeated interaction with Gazebo.

## TODO 3: Training Script

In this task, create a student training script.

The training script connects:

```text
RoboRacer environment + RL algorithm + training parameters
```

A simple DQN training script is created below.

In [ ]:
from pathlib import Path

train_student_code = '''
from stable_baselines3 import DQN
from roboracer_rl.gym_env import RoboRacerEnv

env = RoboRacerEnv()

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-4,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    exploration_fraction=0.2,
    exploration_final_eps=0.05,
    verbose=1,
)

model.learn(total_timesteps=100000)
model.save("models/student_dqn")
'''

Path("train_student.py").write_text(train_student_code)

print("Created train_student.py")

### Important training parameters

| Parameter | Meaning |
|---|---|
| `learning_rate` | Step size used when updating the neural network |
| `buffer_size` | Number of past transitions stored for replay |
| `learning_starts` | Number of steps collected before learning begins |
| `batch_size` | Number of samples used for one update |
| `gamma` | Importance of future rewards |
| `train_freq` | Frequency of training updates |
| `target_update_interval` | How often the target network is updated |
| `exploration_fraction` | How long the agent keeps exploring |
| `exploration_final_eps` | Final probability of random action |

To start training from the notebook:

```python
!python3 train_student.py
```

Or from a terminal:

```bash
cd ~/rrstack/packages
source /opt/ros/humble/setup.bash
source install/setup.bash
python3 train_student.py
```

In [ ]:
# Uncomment this line when Gazebo is running and you are ready to train.
# !python3 train_student.py

## Running the Trained Policy

After training, run the trained policy in Gazebo.

Example command:

```bash
ros2 run roboracer_rl policy_node --model models/student_dqn.zip --algorithm dqn
```

If the executable name is different, check:

```bash
ros2 pkg executables roboracer_rl
```

Then run the correct policy node.

## Model Evaluation Metrics

After training, evaluate the trained model in Gazebo and report simple performance metrics.

The evaluation should include:

- **Total reward**: sum of rewards collected during the evaluation run.
- **Minimum LiDAR distance**: closest distance to a wall or obstacle during the run.
- **Crash status**: whether the car crashed during the run.
- **Lap completion**: whether the car completed one lap around the track.

These metrics are used to judge whether the trained policy is safe and useful. Since this assignment is not focused on racing speed, average speed and episode duration are not used as primary metrics.


In [ ]:
from stable_baselines3 import DQN
from roboracer_rl.gym_env import RoboRacerEnv
import numpy as np

model = DQN.load("models/student_dqn.zip")
env = RoboRacerEnv()

obs, info = env.reset()

total_reward = 0.0
min_lidar_values = []
crashed = False

# Run one evaluation episode
for step in range(300):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += float(reward)
    min_lidar_values.append(info.get("min_lidar", np.nan))

    if terminated:
        crashed = True
        break

    if truncated:
        break

minimum_lidar = float(np.nanmin(min_lidar_values)) if min_lidar_values else float("nan")

print("Evaluation result")
print("-----------------")
print(f"Total reward: {total_reward:.2f}")
print(f"Minimum LiDAR distance: {minimum_lidar:.2f} m")
print(f"Crashed: {crashed}")
print("Lap completed: manually check in Gazebo")

with open("evaluation_result.txt", "w") as f:
    f.write("Evaluation result\n")
    f.write("-----------------\n")
    f.write(f"Total reward: {total_reward:.2f}\n")
    f.write(f"Minimum LiDAR distance: {minimum_lidar:.2f} m\n")
    f.write(f"Crashed: {crashed}\n")
    f.write("Lap completed: manually check in Gazebo\n")

print("\nSaved evaluation_result.txt")


### Deliverable

Submit one ZIP file containing:

```text
student_dqn.zip
train_student.py
evaluation_result.txt
```

The trained model should be loadable using the RoboRacer policy node, and `evaluation_result.txt` should contain the model performance metrics listed above.


## Final Analysis

After running the trained agent, answer these questions:

1. Did the car move?
2. Did the car avoid the walls?
3. Did the car complete a lap?
4. Was the motion smooth or oscillating?
5. Which reward term helped the most?
6. Which reward term caused unwanted behavior?
7. Which training parameter would you tune next?
8. What would you change to improve lap completion?

The goal is not only to train a perfect agent. The goal is to understand how observation design, reward design, action-value estimation, and training affect autonomous driving behavior.

## References

- Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, 2nd Edition.
- OpenAI Spinning Up, *Key Concepts in Reinforcement Learning*.
- Gymnasium documentation, environment `reset()` and `step()` API.
- Stable-Baselines3 documentation for DQN and PPO.